In [ ]:
%pip install seaborn ipykernel

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

# Connect to the SQLite database file we created earlier
# '../' goes up one level since this notebook is inside the 'notebooks' folder
engine = create_engine('sqlite:///../telco_churn.db')

print("Setup complete! Database connection established.")

✅ Setup complete! Database connection established.


In [ ]:
# 1. Write the SQL extraction query
query = """
SELECT * FROM raw_churn;
"""

# 2. Extract into a pandas DataFrame
df = pd.read_sql(query, engine)

# 3. Display the first 5 rows
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [ ]:
# 1. Force convert TotalCharges to numeric values. 
# 'errors="coerce"' tells pandas: "If you see a blank space or text, turn it into a true NaN (missing value)"
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# 2. Check how many missing values were generated by that conversion
print("Missing values per column after conversion:")
print(df.isnull().sum())

Missing values per column after conversion:
customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64


In [ ]:
# 1. Drop rows where TotalCharges is missing
df = df.dropna(subset=['TotalCharges'])

# 2. Drop the customerID column (it's a random string, useless for machine learning)
df = df.drop(columns=['customerID'])

# 3. Double-check that our data shape is correct
print(f"Cleaned dataset shape: {df.shape}")

Cleaned dataset shape: (7032, 20)


In [ ]:
# Create an interactive histogram using Plotly
fig = px.histogram(
    df, 
    x="tenure", 
    color="Churn", 
    barmode="group",
    title="Customer Churn by Tenure (Months)",
    labels={"tenure": "Tenure (Months)", "count": "Number of Customers"},
    color_discrete_map={"No": "#2ecc71", "Yes": "#e74c3c"} # Green for stay, Red for churn
)

fig.update_layout(bargap=0.1)
fig.show()

In [ ]:
# Create an interactive bar chart showing Churn rates across different contracts
fig_contract = px.histogram(
    df, 
    x="Contract", 
    color="Churn", 
    barmode="group",
    title="Customer Churn by Contract Type",
    labels={"Contract": "Contract Type", "count": "Number of Customers"},
    color_discrete_map={"No": "#2ecc71", "Yes": "#e74c3c"}
)

fig_contract.show()

In [ ]:
# 1. Convert the target column 'Churn' into binary (1 for Yes, 0 for No)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# 2. Automatically convert all other text columns into numeric flags (0 or 1)
# drop_first=True avoids the "dummy variable trap" by dropping redundant columns
df_encoded = pd.get_dummies(df, drop_first=True)

# 3. View our new, entirely numeric dataset
print(f"Encoded dataset shape: {df_encoded.shape}")
df_encoded.head()

Encoded dataset shape: (7032, 31)


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,False,True,False,False,True,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,0,True,False,False,True,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,1,True,False,False,True,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,0,True,False,False,False,True,...,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,1,False,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False
